# 02 Grid and Isochrones
This notebook builds the Shanghai 500 m grid and computes or approximates 15-minute reachable areas for walk, bike, transit, and car. The default method is a transparent buffer approximation calibrated by mode speed. If a routing API is supplied later, the `isochrones.py` module can be extended without changing downstream scoring.

## 0. Setup

In [ ]:
from pathlib import Path
import sys
import geopandas as gpd
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from fifteenmc.paths import load_project_paths, load_yaml
from fifteenmc.cleaning import load_boundary, normalize_green
from fifteenmc.grid import build_grid, attach_districts
from fifteenmc.io import write_geoparquet
from fifteenmc.isochrones import build_buffer_isochrones, build_accessibility_table, nearest_distance

paths = load_project_paths(PROJECT_ROOT)
weights = load_yaml(PROJECT_ROOT / 'config' / 'weights.yaml')

## 1. Load Shanghai Boundary
The grid is clipped to the study boundary and uses EPSG:32651 for meter-based cell construction.

In [ ]:
boundary = load_boundary(paths)
boundary.head()

## 2. Construct 500 m Grid

In [ ]:
grid = build_grid(boundary, grid_size_m=paths.config['analysis']['grid_size_m'])
grid = attach_districts(grid, boundary)
write_geoparquet(grid, paths.output('grid_500m'))
print(grid.shape)
grid.head()

## 3. Build Four-Mode 15-Minute Isochrones
For reproducibility without paid services, the MVP uses speed-based buffer isochrones: walk 4.8 km/h, bike 14 km/h, transit 18 km/h, car 28 km/h. These are documented approximations, not turn-by-turn network routes.

In [ ]:
isochrones = {}
for mode, cfg in weights['modes'].items():
    cfg = dict(cfg)
    cfg['mode'] = mode
    isochrones[mode] = build_buffer_isochrones(grid, cfg)
    print(mode, round(isochrones[mode]['radius_m'].iloc[0], 1), 'm')

## 4. Spatial Joins for Accessibility Metrics
Amenities and transit stops are counted inside each isochrone. Green polygons are summarized as green area within the 15-minute reachable area. Dedicated cycling lane length can be added once a source field identifying cycle lanes is confirmed.

In [ ]:
amenities = gpd.read_parquet(paths.output('amenities'))
green = normalize_green(paths)
access = build_accessibility_table(grid, amenities, green, weights)
print(access.shape)
access.head()

## 5. Nearest Metro Distance
The web app detail panel requires metro distance. This is calculated from grid cell centroid to the nearest metro station or exit.

In [ ]:
transit = gpd.read_parquet(paths.output('transit'))
metro = transit[transit['category_group'].isin(['metro_station', 'metro_exit'])].copy()
distance = nearest_distance(grid, metro, 'metro_distance_m')
access = access.merge(distance, on='grid_id', how='left')
access['cycle_lane_km'] = 0.0
access.to_parquet(paths.output('grid_accessibility'), index=False)
print(paths.output('grid_accessibility'))
access.head()

## 6. Cache Summary
All expensive spatial joins are cached to `data/processed/grid_accessibility.parquet`, so scoring and web export can be rerun without rebuilding isochrone joins.